# Frozen-feature diagnostic probes

Fits small, fixed logistic probes on frozen DINOv3 features. DINOv3 is never trained.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

from pathlib import Path
import pandas as pd

from notebook_config import (
    DATA_ROOT,
    HF_MODEL_SOURCE,
    PROBE_CONTROL_REPEATS,
    RUN_ROOT,
)
from notebook_support import ensure_core_outputs, run_once

audit_metadata, legacy_manifest = ensure_core_outputs(RUN_ROOT)
grouped_manifest = RUN_ROOT / "grouped_protocol" / "grouped_fold_manifest.csv"
if grouped_manifest.is_file():
    manifest = grouped_manifest
    output_suffix = "_grouped"
    print("Using leakage-aware grouped folds created by notebook 04.")
else:
    manifest = legacy_manifest
    output_suffix = ""
    print("Using reconstructed historical image-level folds; no verified grouping manifest is available.")
recovered_features = RUN_ROOT / "recovered_concat" / "frozen_cls_features.npz"

if recovered_features.is_file():
    feature_file = recovered_features
    print("Using frozen CLS features recovered from the validated concat checkpoints.")
else:
    import torch
    from huggingface_hub import snapshot_download
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = 32 if device == "cuda" else 8
    model_source = Path(HF_MODEL_SOURCE)
    if not model_source.is_dir():
        try:
            model_source = Path(snapshot_download(
                repo_id=HF_MODEL_SOURCE,
                allow_patterns=["*.json", "*.safetensors"],
                local_files_only=True,
            ))
        except Exception as error:
            raise RuntimeError(
                "The pretrained DINOv3 snapshot is not cached. Run "
                "02A_DINOv3_One_Time_Cache.ipynb first."
            ) from error
    cache_dir = run_once(
        "08_cache_frozen_features.py",
        [
            "--model-path", str(model_source),
            "--metadata", str(audit_metadata),
            "--data-root", str(DATA_ROOT),
            "--batch-size", str(batch_size),
            "--device", device,
        ],
        RUN_ROOT / "feature_cache",
        ["frozen_cls_features.npz", "feature_provenance.json"],
    )
    feature_file = cache_dir / "frozen_cls_features.npz"
    print("Cached CLS features with one local pretrained-DINOv3 inference pass; no DINOv3 training.")

probe_dir = run_once(
    "05_train_frozen_probes.py",
    [
        "--features", str(feature_file),
        "--metadata", str(audit_metadata),
        "--manifest", str(manifest),
        "--confirm-independent-frozen-features",
        "--train-control-repeats", str(PROBE_CONTROL_REPEATS),
    ],
    RUN_ROOT / f"probes{output_suffix}",
    ["probe_oof_predictions.csv", "probe_fold_metrics.csv", "classes.json", "protocol.json"],
)

evaluation = run_once(
    "03_evaluate_predictions.py",
    [
        "--predictions", str(probe_dir / "probe_oof_predictions.csv"),
        "--classes", str(probe_dir / "classes.json"),
        "--metadata", str(audit_metadata),
        "--manifest", str(manifest),
        "--compare", "probe_concat,probe_image_only",
    ],
    RUN_ROOT / f"probe_evaluation{output_suffix}",
    ["overall.csv", "disease_metrics.csv", "support_sensitivity.csv", "paired_comparison.json"],
)

overall = pd.read_csv(evaluation / "overall.csv")
preferred = ["probe_image_only", "probe_metadata_only", "probe_concat"]
base = overall[overall["model"].isin(preferred)].copy()
base["order"] = base["model"].map({name: index for index, name in enumerate(preferred)})
print(base.sort_values("order").drop(columns="order").to_string(index=False))
print("\nThese are new fixed logistic probes, not retrained DSAF heads. DINOv3 remains frozen.")
